<a href="https://colab.research.google.com/github/bahulmishra/Image_captioner/blob/main/IMAGE_CAPTIONER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMAGE CAPTION GENERATOR


##dependencies


In [ ]:
import os
import re
import math
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, add
from tensorflow.keras.layers import Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from tensorflow.keras.utils import plot_model
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm_notebook
from collections import Counter
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import pickle
from tensorflow.keras.models import load_model

In [ ]:
from google.colab import drive

# Mount Drive once for the entire notebook
drive.mount('/content/drive')

# Set the path variable once for the entire notebook
os.environ['MY_DRIVE_PATH'] = '/content/drive/MyDrive/projects/image_captioner'

In [ ]:
import kagglehub
path = kagglehub.dataset_download("adityajn105/flickr8k")

In [ ]:
images_directory = os.path.join(path, 'Images')
captions_path = os.path.join(path, 'captions.txt')
def load_captions(file_path):
    with open(file_path, 'r') as f:
        captions = f.readlines()
        captions = [caption.lower() for caption in captions[1:]]
    return captions

def tokenize_captions(captions):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(captions)
    return tokenizer

captions = load_captions(captions_path)
captions[:15:3]

## DENOISING


In [ ]:
def clean_text(text):
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_captions = [clean_text(caption.split(',')[1]) for caption in captions]
cleaned_captions[:15:2]

In [ ]:
# Cleaning the captions:  process the captions by adding start and end tokens to define the sentence boundaries

captions_IDs = []
for i in range(len(cleaned_captions)):
    item = captions[i].split(',')[0]+'\t'+'start '+cleaned_captions[i]+' end\n'
    captions_IDs.append(item)

captions_IDs[:20:3], len(captions_IDs)


In [ ]:
# Loading images with captions: Mapping the images to the captions

def visualaization(data, num_of_images):
    captions_dictionary = {}
    for item in data[100:100+(num_of_images)*5]:
        image_id, caption = item.split('\t')
        if image_id not in captions_dictionary:
            captions_dictionary[image_id] = []
        captions_dictionary[image_id].append(caption)
    else:
        list_captions = [x for x in captions_dictionary.items()]

    count = 1
    fig = plt.figure(figsize=(10,20))
    for filename in list(captions_dictionary.keys()):
        captions = captions_dictionary[filename]

        img_path = os.path.join(images_directory, filename.strip())
        image_load = load_img(img_path, target_size=(199, 199, 3))

        ax = fig.add_subplot(num_of_images,2,count,xticks=[],yticks=[])
        ax.imshow(image_load)
        count += 1

        ax = fig.add_subplot(num_of_images,2,count)
        plt.axis('off')
        ax.plot()
        ax.set_xlim(0,1)
        ax.set_ylim(0,len(captions))
        for i, caption in enumerate(captions):
            ax.text(0,i,caption,fontsize=20)
        count += 1
    plt.show()

visualaization(captions_IDs, 5)

In [ ]:
# Analyze the length of captions to determine an optimal sequence length.

def captions_length(data):
    plt.figure(figsize=(15, 7), dpi=300)
    sns.set_style('darkgrid')
    sns.histplot(x=[len(x.split(' ')) for x in data], kde=True, binwidth=1)
    plt.title('Captions length histogram', fontsize=15, fontweight='bold')
    plt.xticks(fontweight='bold')
    plt.yticks(fontweight='bold')
    plt.xlabel('Length', fontweight='bold')
    plt.ylabel('Freaquency', fontweight='bold')
    plt.show()

captions_length(cleaned_captions)

In [ ]:
# Tokenising the vocab


temp_tokenizer = Tokenizer()
temp_tokenizer.fit_on_texts(cleaned_captions)

# how many words appear 4 times or more
threshold = 4
frequent_words_count = sum(1 for count in temp_tokenizer.word_counts.values() if count >= threshold)


tokenizer = Tokenizer(num_words=frequent_words_count + 1, oov_token="<unk>")
tokenizer.fit_on_texts(cleaned_captions)

vocab_size = frequent_words_count + 1

print(f"Original Vocab Size: {len(temp_tokenizer.word_index)}")
print(f"New Optimized Vocab Size: {vocab_size}")

In [ ]:
#  train, validation and test splits

all_image_ids = os.listdir(images_directory)

train_image_ids, val_image_ids = train_test_split(all_image_ids, test_size=0.15, random_state=42)
val_image_ids, test_image_ids = train_test_split(val_image_ids, test_size=0.1, random_state=42)

train_captions, val_captions, test_captions = [], [], []
for caption in captions_IDs:
    image_id, _ = caption.split('\t')

    if image_id in train_image_ids:
        train_captions.append(caption)

    elif image_id in val_image_ids:
        val_captions.append(caption)

    elif image_id in test_image_ids:
        test_captions.append(caption)

    else:
        print('Unknown image ID !')

train_captions[0], val_captions[0], test_captions[0], len(train_captions)/5, len(val_captions)/5, len(test_captions)/5


## Encoding


In [ ]:
#  To convert the image to encoding ( we are using nceptionV3 model which has been trained on Imagenet dataset that had 1000 different classes to classify)

def preprocess_image(image_path):
    img = load_img(image_path, target_size=(299, 299))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return img

def extract_image_features(model, image_path):
    img = preprocess_image(image_path)
    features = model.predict(img, verbose=0)
    return features

inception_v3_model = InceptionV3(weights = 'imagenet', input_shape=(299, 299, 3))
inception_v3_model.layers.pop()
inception_v3_model = Model(inputs=inception_v3_model.inputs, outputs=inception_v3_model.layers[-2].output)

In [ ]:
# Extracting Image Features for Training, Validation and Testing

train_image_features, val_image_features, test_image_features = {}, {}, {}  # A Dictionary to store image features with their corresponding IDs

pbar = tqdm_notebook(total=len(all_image_ids), position=0, leave=True, colour='green')

for caption in all_image_ids:
    image_id = caption.split('\t')[0]
    image_path = os.path.join(images_directory, image_id)
    image_features = extract_image_features(inception_v3_model, image_path) # Extracting features

    if image_id in train_image_ids:
        train_image_features[image_id] = image_features.flatten()  # Flattening the features
        pbar.update(1)

    elif image_id in val_image_ids:
        val_image_features[image_id] = image_features.flatten()  # Flattening the features
        pbar.update(1)

    elif image_id in test_image_ids:
        test_image_features[image_id] = image_features.flatten()  # Flattening the features
        pbar.update(1)

    else:
        print('Unknown image ID !')

pbar.close()

In [ ]:

save_path = os.environ.get('MY_DRIVE_PATH')
os.makedirs(save_path, exist_ok=True)

# Saving the extracted features to your Drive
with open(os.path.join(save_path, 'train_features.pkl'), 'wb') as f:
    pickle.dump(train_image_features, f)
with open(os.path.join(save_path, 'val_features.pkl'), 'wb') as f:
    pickle.dump(val_image_features, f)
with open(os.path.join(save_path, 'test_features.pkl'), 'wb') as f:
    pickle.dump(test_image_features, f)

print("Image features successfully saved to Drive!")

## Model Training


In [ ]:
#Data Generator for Model Training

def data_generator(captions, image_features, tokenizer, max_caption_length, batch_size):
    num_samples = len(captions)
    image_ids = list(image_features.keys())
    while True:
        np.random.shuffle(image_ids)  # Shuffle image_ids for each epoch
        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            X_images, X_captions, y = [], [], []
            for caption in captions[start_idx:end_idx]:
                image_id, caption_text = caption.split('\t')
                caption_text = caption_text.rstrip('\n')
                seq = tokenizer.texts_to_sequences([caption_text])[0] # Tokenizing the caption
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i] # X_caption, Y
                    in_seq = pad_sequences([in_seq], maxlen=max_caption_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    X_images.append(image_features[image_id])
                    X_captions.append(in_seq)
                    y.append(out_seq)

            yield (np.array(X_images), np.array(X_captions)), np.array(y)


max_caption_length = max(len(caption.split()) for caption in cleaned_captions) + 1

cnn_output_dim = inception_v3_model.output_shape[1] # 2048

batch_size_train = 270
batch_size_val = 150

train_data_generator = data_generator(train_captions, train_image_features, tokenizer, max_caption_length, batch_size_train)
val_data_generator = data_generator(val_captions, val_image_features, tokenizer, max_caption_length, batch_size_val)

In [ ]:
# Building ImageCaptioning Model

"""
Architecture

1. The Vision Branch:
   We take the massive 2048-number summary of the image from our InceptionV3 model
   and compress it down to 128 essential features. So, our model squints
   to focus only on the most important shapes and colors in the picture.

2. The Language Branch:
   Now, we take our tokenized words and map them into a 128-dimensional space, then pass
   them through an LSTM (a memory network). We added a 30% Dropout here, meaning we
   randomly blindfold 30% of the neurons every single time the model studies a sentence.
   This forces the model to actually learn the underlying grammar instead of just
   memorizing the exact phrases from the training data.

3. The Decoder:
   Here,we add the compressed Vision features and the language features together. Now the
   model is simultaneously looking at the picture and remembering the current sentence.
   Finally, it uses a Dense layer to calculate the logical next word out of our
   3,319-word dictionary.
"""

def build_model(vocab_size, max_caption_length, cnn_output_dim):
    # Vision Branch
    input_image = Input(shape=(cnn_output_dim,), name='Features_Input')
    fe1 = BatchNormalization()(input_image)
    fe2 = Dense(128, activation='relu')(fe1) # Halved to 128 for faster training
    fe3 = BatchNormalization()(fe2)

    # Language Branch
    input_caption = Input(shape=(max_caption_length,), name='Sequence_Input')
    se1 = Embedding(vocab_size, 128, mask_zero=True)(input_caption) # Halved to 128
    se1_drop = Dropout(0.3)(se1)               # Goldilocks Dropout: 0.3
    se2 = LSTM(128)(se1_drop)                  # Halved to 128
    se2_drop = Dropout(0.3)(se2)               # Goldilocks Dropout: 0.3

    # Decoder
    decoder1 = add([fe3, se2_drop])
    decoder2 = Dense(128, activation='relu')(decoder1) # Halved to 128
    outputs = Dense(vocab_size, activation='softmax', name='Output_Layer')(decoder2)

    model = Model(inputs=[input_image, input_caption], outputs=outputs, name='Image_Captioning')
    return model

In [ ]:

caption_model = build_model(vocab_size, max_caption_length, cnn_output_dim)

my_optimizer = Adam(learning_rate=0.001)
caption_model.compile(loss='categorical_crossentropy', optimizer= my_optimizer)

caption_model.summary()

In [ ]:
plot_model(caption_model, show_shapes=True, show_layer_names=True)

In [ ]:
# Training Model ( with early stopping and Learning Rate Scheduling)

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# decay by 5% to maintain momentum with the new Dropout
def lr_scheduler(epoch, lr):
    return lr * 0.95

lr_schedule = LearningRateScheduler(lr_scheduler)

history = caption_model.fit(train_data_generator,
                            steps_per_epoch=len(train_captions) // batch_size_train,
                            validation_data=val_data_generator,
                            validation_steps=len(val_captions) // batch_size_val,
                            epochs=25,
                            callbacks=[early_stopping, lr_schedule])

In [ ]:
# loss curve:

plt.figure(figsize=(15, 7), dpi=200)
sns.set_style('whitegrid')
plt.plot([x+1 for x in range(len(history.history['loss']))], history.history['loss'], color='#E74C3C', marker='o')
plt.plot([x+1 for x in range(len(history.history['loss']))], history.history['val_loss'], color='#641E16', marker='h')
plt.title('Train VS Validation', fontsize=15, fontweight='bold')
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Loss', fontweight='bold')
plt.legend(['Train Loss', 'Validation Loss'], loc='best')
plt.show()

In [ ]:

save_path = os.environ.get('MY_DRIVE_PATH')
os.makedirs(save_path, exist_ok=True)

# Saving the model and tokenizer
caption_model.save(os.path.join(save_path, 'imagecaptioner_weights_tokens.h5'))
with open(os.path.join(save_path, 'imagecaptioner_weights_tokens.pkl'), 'wb') as file:
    pickle.dump(tokenizer, file)

print(f"Model and Tokenizer successfully saved to: {save_path}")

In [ ]:
with open(os.path.join(save_path, 'imagecaptioner_weights_tokens.pkl'), 'rb') as file:
    tokenizer = pickle.load(file)

vocab_size = tokenizer.num_words
max_caption_length = 34
cnn_output_dim = 2048

caption_model = build_model(vocab_size, max_caption_length, cnn_output_dim)

caption_model.load_weights(os.path.join(save_path, 'imagecaptioner_weights_tokens.h5'))


## Caption Generation

In [ ]:
# Greedy Search

def greedy_generator_with_penalty(image_features, repetition_penalty=0.1):
    in_text = 'start '
    generated_indices = set()

    # Define grammatical words that are ALLOWED to repeat
    stop_words = ['a', 'the', 'in', 'on', 'and', 'with', 'is', 'to', 'of', 'an', 'at']
    # Look up their index in your tokenizer so we can skip penalizing them
    safe_indices = {tokenizer.word_index[w] for w in stop_words if w in tokenizer.word_index}

    for _ in range(max_caption_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_caption_length).reshape((1, max_caption_length))
        prediction = caption_model.predict([image_features.reshape(1, cnn_output_dim), sequence], verbose=0)[0]

        # Apply the repetition penalty ONLY to descriptive words
        for idx in generated_indices:
            if idx not in safe_indices:
                prediction[idx] *= repetition_penalty

        idx = np.argmax(prediction)
        word = tokenizer.index_word.get(idx)

        if word is None:
            break

        generated_indices.add(idx)
        in_text += ' ' + word

        if word == 'end':
            break

    in_text = in_text.replace('start ', '')
    in_text = in_text.replace(' end', '')
    return in_text


In [ ]:
# Beam search

def beam_search_generator_with_penalty(image_features, K_beams=3, repetition_penalty=0.1):
    start = [tokenizer.word_index['start']]
    start_word = [[start, 0.0]]

    # Safe words that can repeat
    stop_words = ['a', 'the', 'in', 'on', 'and', 'with', 'is', 'to', 'of', 'an', 'at']
    safe_indices = {tokenizer.word_index[w] for w in stop_words if w in tokenizer.word_index}

    for _ in range(max_caption_length):
        temp = []
        for s in start_word:
            sequence = pad_sequences([s[0]], maxlen=max_caption_length).reshape((1, max_caption_length))
            preds = caption_model.predict([image_features.reshape(1, cnn_output_dim), sequence], verbose=0)[0]

            # Apply the repetition penalty to words already in THIS beam's sequence
            for idx in set(s[0]):
                if idx not in safe_indices and idx < len(preds):
                    preds[idx] *= repetition_penalty

            word_preds = np.argsort(preds)[-K_beams:]

            for w in word_preds:
                next_cap, prob = s[0][:], s[1]
                next_cap.append(w)
                prob += preds[w] # Adding probability
                temp.append([next_cap, prob])

        start_word = temp
        start_word = sorted(start_word, reverse=False, key=lambda l: l[1])
        start_word = start_word[-K_beams:]

    start_word = start_word[-1][0]
    captions_ = [tokenizer.index_word.get(i, '') for i in start_word]

    final_caption = []
    for i in captions_:
        if i != 'end':
            final_caption.append(i)
        else:
            break

    final_caption = ' '.join(final_caption[1:])
    return final_caption

In [ ]:
# BLEU score


def BLEU_score(actual, greedy, beam_search):
    actual_tokens = [sentence.split() for sentence in actual]
    greedy_tokens = greedy[0].split()
    beam_search_tokens = beam_search[0].split()

    refs = [actual_tokens]
    hyp_greedy = [greedy_tokens]
    hyp_beam = [beam_search_tokens]

    smoothie = SmoothingFunction().method4

    # BLEU-1 (100% weight to 1-grams)
    score_greedy_1 = corpus_bleu(refs, hyp_greedy, weights=(1.0, 0, 0, 0), smoothing_function=smoothie)
    score_BS_1 = corpus_bleu(refs, hyp_beam, weights=(1.0, 0, 0, 0), smoothing_function=smoothie)

    # BLEU-2 (50% weight to 1-grams, 50% weight to 2-grams)
    score_greedy_2 = corpus_bleu(refs, hyp_greedy, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    score_BS_2 = corpus_bleu(refs, hyp_beam, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)

    return [
         (f'BLEU-2 Greedy: {round(score_greedy_2, 5)}'),
         (f'BLEU-1 Greedy: {round(score_greedy_1, 5)}'),
         (f'Greedy: {greedy[0]}'),
         (f'BLEU-2 Beam Search: {round(score_BS_2, 5)}'),
         (f'BLEU-1 Beam Search: {round(score_BS_1, 5)}'),
         (f'Beam Search:  {beam_search[0]}')
    ]

In [ ]:
# generating test captions

generated_captions = {}

pbar = tqdm_notebook(total=len(test_image_features), position=0, leave=True, colour='green')
for image_id in test_image_features:
    cap = greedy_generator_with_penalty(test_image_features[image_id])
    generated_captions[image_id] = cap
    pbar.update(1)

pbar.close()

In [ ]:
def visualization(data, greedy_caps, beamS_generator, evaluator, num_of_images):

    captions_dictionary = {}
    for item in data:
        image_id, caption = item.split('\t')
        if image_id not in captions_dictionary:
            captions_dictionary[image_id] = []
        captions_dictionary[image_id].append(caption.strip())


    keys = list(captions_dictionary.keys())
    images = [np.random.choice(keys) for i in range(num_of_images)]
    count = 1
    fig = plt.figure(figsize=(6,20))

    for filename in images:
        actual_cap = captions_dictionary[filename]
        actual_cap = [x.replace("start ", "") for x in actual_cap]
        actual_cap = [x.replace(" end", "") for x in actual_cap]

        greedy_cap = greedy_caps[filename]
        beamS_cap = beamS_generator(test_image_features[filename], K_beams=5)


        caps_with_score = evaluator(actual_cap, [greedy_cap]*(len(actual_cap)), [beamS_cap]*(len(actual_cap)))


        img_path = os.path.join(images_directory, filename.strip())
        image_load = load_img(img_path, target_size=(199,199,3))

        ax = fig.add_subplot(num_of_images,2,count,xticks=[],yticks=[])
        ax.imshow(image_load)
        count += 1

        ax = fig.add_subplot(num_of_images,2,count)
        plt.axis('off')
        ax.plot()
        ax.set_xlim(0,1)
        ax.set_ylim(0,len(caps_with_score))
        for i, text in enumerate(caps_with_score):
            ax.text(0,i,text,fontsize=10)
        count += 1
    plt.show()

visualization(test_captions, generated_captions, beam_search_generator_with_penalty, BLEU_score, 5)

In [ ]:
visualization(test_captions, generated_captions, beam_search_generator_with_penalty, BLEU_score, 5)

In [ ]:
def show_captions(data, generated_caps, num_images=5):

    captions_dictionary = {}
    for item in data:
        image_id, caption = item.split('\t')
        if image_id not in captions_dictionary:
            captions_dictionary[image_id] = []

        clean_cap = caption.replace("start ", "").replace(" end", "").strip()
        captions_dictionary[image_id].append(clean_cap)

    keys = list(captions_dictionary.keys())
    selected_images = np.random.choice(keys, size=num_images, replace=False)

    fig, axes = plt.subplots(num_images, 2, figsize=(16, 4 * num_images),
                             gridspec_kw={'width_ratios': [1, 2]})


    if num_images == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, filename in enumerate(selected_images):

        img_path = os.path.join(images_directory, filename.strip())
        image_load = load_img(img_path, target_size=(299, 299, 3))

        axes[i, 0].imshow(image_load)
        axes[i, 0].axis('off')


        actual_caps = captions_dictionary[filename]
        gen_cap = generated_caps.get(filename, "No caption generated")


        text_display = f"GENERATED CAPTION:\n{gen_cap}\n\n"
        text_display += "ORIGINAL CAPTIONS:\n" + "\n".join([f"• {cap}" for cap in actual_caps])


        axes[i, 1].text(0.05, 0.5, text_display, fontsize=14, va='center', ha='left', wrap=True,
                        bbox=dict(boxstyle="round,pad=0.8", facecolor="#f8f9fa", edgecolor="#dee2e6", alpha=0.9))
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.05)
    plt.show()

In [ ]:
show_captions(test_captions, generated_captions, num_images=5)